# theOdds API – Test Notebook
Tests connectivity and core endpoints for the [The Odds API](https://the-odds-api.com/liveapi/guides/v4/).

**Endpoints covered:**
1. `/sports` – list in-season sports
2. `/sports/basketball_ncaab/odds` – **live events only** (games already in progress)
3. Implied probability & vig removal

In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), '.env'))

API_KEY = os.getenv('ODDS_API_KEY')
BASE_URL = 'https://api.the-odds-api.com/v4'

assert API_KEY, 'ODDS_API_KEY not found in .env'
print(f'API key loaded: {API_KEY[:6]}...{API_KEY[-4:]}')

API key loaded: b1098b...e124


## 1. List In-Season Sports

In [2]:
resp = requests.get(
    f'{BASE_URL}/sports',
    params={'apiKey': API_KEY}
)
resp.raise_for_status()

print(f'Requests used: {resp.headers.get("x-requests-used")} / {resp.headers.get("x-requests-remaining")} remaining')

sports = resp.json()
df_sports = pd.DataFrame(sports)
print(f'\n{len(df_sports)} sports available')
df_sports[['key', 'group', 'title', 'active']]

Requests used: 248 / 252 remaining

87 sports available


,key,group,title,active
0,americanfootball_ncaaf,American Football,NCAAF,True
1,americanfootball_ncaaf_championship_winner,American Football,NCAAF Championship Winner,True
2,americanfootball_nfl_super_bowl_winner,American Football,NFL Super Bowl Winner,True
3,americanfootball_ufl,American Football,UFL,True
4,aussierules_afl,Aussie Rules,AFL,True
...,...,...,...,...
82,soccer_uefa_champs_league,Soccer,UEFA Champions League,True
83,soccer_uefa_europa_conference_league,Soccer,UEFA Europa Conference League,True
84,soccer_uefa_europa_league,Soccer,UEFA Europa League,True
85,soccer_usa_mls,Soccer,MLS,True


## 2. Fetch Live NCAAB Odds
Filters to games that have **already started** (`commenceTimeTo=now`) but haven't been running longer than 4 hours (`commenceTimeFrom=now-4h`), which excludes completed games. Uses Pinnacle + DraftKings, h2h market.

In [3]:
from datetime import datetime, timezone, timedelta

SPORT = 'basketball_ncaab'

now_utc = datetime.now(timezone.utc)
lookahead_hours = 3   # adjust: how far ahead to look for upcoming games

soon = now_utc + timedelta(hours=lookahead_hours)

# commenceTimeFrom=now  → exclude games already started
# commenceTimeTo=now+Xh → only games starting within the next X hours
params = {
    'apiKey': API_KEY,
    'regions': 'us',
    'markets': 'h2h',
    'oddsFormat': 'decimal',
    'bookmakers': 'pinnacle,draftkings',
    'commenceTimeFrom': now_utc.strftime('%Y-%m-%dT%H:%M:%SZ'),
    'commenceTimeTo':   soon.strftime('%Y-%m-%dT%H:%M:%SZ'),
}

print(f'Sport          : {SPORT}')
print(f'commenceTimeFrom: {params["commenceTimeFrom"]}')
print(f'commenceTimeTo  : {params["commenceTimeTo"]}')

resp_odds = requests.get(f'{BASE_URL}/sports/{SPORT}/odds', params=params)
resp_odds.raise_for_status()

print(f'\nRequests used: {resp_odds.headers.get("x-requests-used")} / {resp_odds.headers.get("x-requests-remaining")} remaining')

events = resp_odds.json()
print(f'{len(events)} upcoming event(s) in the next {lookahead_hours}h')
for e in events:
    print(f"  {e['away_team']} @ {e['home_team']}  —  tips off {e['commence_time']}")

Sport          : basketball_ncaab
commenceTimeFrom: 2026-04-06T20:02:58Z
commenceTimeTo  : 2026-04-06T23:02:58Z

Requests used: 248 / 252 remaining
0 upcoming event(s) in the next 3h


## 3. Parse Raw Response

In [4]:
if events:
    import json
    print(json.dumps(events[0], indent=2))
else:
    print('No events returned for this sport right now.')

No events returned for this sport right now.


## 4. Implied Probability & Vig Removal
For a two-outcome market:
- **Implied prob** = 1 / decimal_odds
- **Overround (vig)** = sum of implied probs – 1
- **Fair prob** = implied_prob / sum_of_implied_probs

In [5]:
rows = []

for event in events:
    for book in event.get('bookmakers', []):
        for market in book.get('markets', []):
            if market['key'] != 'h2h':
                continue
            outcomes = market['outcomes']
            implied = [1 / o['price'] for o in outcomes]
            overround = sum(implied) - 1
            fair_probs = [p / sum(implied) for p in implied]
            for o, imp, fair in zip(outcomes, implied, fair_probs):
                rows.append({
                    'event_id': event['id'],
                    'home': event['home_team'],
                    'away': event['away_team'],
                    'commence': event['commence_time'],
                    'bookmaker': book['key'],
                    'outcome': o['name'],
                    'decimal_odds': o['price'],
                    'implied_prob': round(imp, 4),
                    'vig_pct': round(overround * 100, 3),
                    'fair_prob': round(fair, 4),
                })

df = pd.DataFrame(rows)
print(f'{len(df)} outcome rows')
df.head(20)

0 outcome rows


""


## 5. Pinnacle Fair Odds (no-vig)
These are the target probabilities to compare against Kalshi.

In [6]:
if not df.empty:
    pinnacle = df[df['bookmaker'] == 'pinnacle'][['home', 'away', 'commence', 'outcome', 'decimal_odds', 'fair_prob', 'vig_pct']]
    if pinnacle.empty:
        print('No Pinnacle lines returned — they may not be available in this region/sport. Showing all bookmakers.')
        print(df[['bookmaker', 'outcome', 'decimal_odds', 'fair_prob', 'vig_pct']].drop_duplicates())
    else:
        print(pinnacle.to_string(index=False))
else:
    print('No data to display.')

No data to display.


## 6. API Quota Check

In [7]:
print('Remaining quota after this session:')
print(f'  Requests remaining : {resp_odds.headers.get("x-requests-remaining")}')
print(f'  Requests used      : {resp_odds.headers.get("x-requests-used")}')
print(f'  Last update        : {resp_odds.headers.get("x-timestamp")}')

Remaining quota after this session:
  Requests remaining : 252
  Requests used      : 248
  Last update        : None
